# 03 Incrementality Simulation

This notebook turns the CRM experiment design into a commercial decision model. It estimates whether a second purchase accelerator campaign could create positive net incremental value after accounting for natural repeat behaviour, contribution margin, discount cost, campaign cost, and cannibalisation risk.

Inputs:

- `../data/processed/customer_metrics.parquet`
- Customer order logic rebuilt from `../data/processed/clean_transactions.parquet`

Outputs:

- `../data/processed/incrementality_scenarios.csv`
- `../data/processed/incrementality_sensitivity.csv`


## Setup

The model uses current retention analysis outputs to size the opportunity and set directional baseline assumptions. Because there is no real campaign exposure data in this public dataset, the simulation is scenario-based rather than a measured causal result.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")
CUSTOMER_METRICS_PATH = PROCESSED_DIR / "customer_metrics.parquet"
TRANSACTIONS_PATH = PROCESSED_DIR / "clean_transactions.parquet"

for path in [CUSTOMER_METRICS_PATH, TRANSACTIONS_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found: {path}. Run notebooks 01 and 02 first."
        )

## Baseline Inputs

The model uses observed customer behaviour to create a directional baseline: total valid customers, one-time customers, observed second purchase conversion, and average second-order value.

In [ ]:
customer_metrics = pd.read_parquet(CUSTOMER_METRICS_PATH)
transactions = pd.read_parquet(TRANSACTIONS_PATH)

transactions.columns = (
    transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
transactions = transactions.rename(
    columns={
        "invoiceno": "invoice_no",
        "invoicedate": "invoice_date",
        "customerid": "customer_id",
    }
)
transactions["invoice_date"] = pd.to_datetime(transactions["invoice_date"])

valid_lines = transactions[transactions["is_valid_order_line"]].copy()
customer_orders = (
    valid_lines.groupby(["customer_id", "invoice_no"], as_index=False)
    .agg(
        order_date=("invoice_date", "min"),
        order_revenue=("analytical_revenue", "sum"),
    )
)
customer_orders = customer_orders[customer_orders["order_revenue"] > 0].copy()
customer_orders = customer_orders.sort_values(
    ["customer_id", "order_date", "invoice_no"]
).reset_index(drop=True)
customer_orders["order_rank"] = (
    customer_orders.groupby("customer_id").cumcount() + 1
)

total_customers = len(customer_metrics)
repeat_customers = int(customer_metrics["is_repeat_customer"].sum())
one_time_customers = total_customers - repeat_customers
observed_second_purchase_conversion = repeat_customers / total_customers

second_orders = customer_orders[customer_orders["order_rank"] == 2].copy()
average_second_order_value = second_orders["order_revenue"].mean()
median_second_order_value = second_orders["order_revenue"].median()

baseline = pd.Series(
    {
        "total_valid_customers": total_customers,
        "repeat_customers": repeat_customers,
        "one_time_customers": one_time_customers,
        "observed_second_purchase_conversion": observed_second_purchase_conversion,
        "average_second_order_value": average_second_order_value,
        "median_second_order_value": median_second_order_value,
    }
).to_frame("value")

baseline

## Scenario Assumptions

The simulation sizes a future MVP campaign using the current one-time customer count as a practical audience proxy. The observed second purchase conversion is used as the directional natural baseline. Treatment conversion, margin, discount cost, operating cost, and pull-forward risk vary by scenario.

In [ ]:
eligible_customers = one_time_customers
control_conversion = observed_second_purchase_conversion
second_order_aov = average_second_order_value

scenario_assumptions = pd.DataFrame(
    [
        {
            "scenario": "Conservative",
            "treatment_conversion": 0.670,
            "contribution_margin_rate": 0.35,
            "discount_cost_per_order": 6.00,
            "campaign_cost_per_customer": 0.10,
            "pull_forward_rate": 0.40,
        },
        {
            "scenario": "Base",
            "treatment_conversion": 0.700,
            "contribution_margin_rate": 0.40,
            "discount_cost_per_order": 4.00,
            "campaign_cost_per_customer": 0.08,
            "pull_forward_rate": 0.25,
        },
        {
            "scenario": "Optimistic",
            "treatment_conversion": 0.730,
            "contribution_margin_rate": 0.45,
            "discount_cost_per_order": 2.00,
            "campaign_cost_per_customer": 0.06,
            "pull_forward_rate": 0.10,
        },
    ]
)

scenario_assumptions

## Incrementality Model

The model starts with treatment versus control conversion uplift, then converts that uplift into incremental orders and revenue. It reduces value for estimated pull-forward behaviour and subtracts discount and campaign costs.

In [ ]:
def calculate_incrementality(row):
    incremental_conversion = row["treatment_conversion"] - control_conversion
    incremental_orders = eligible_customers * incremental_conversion
    gross_incremental_revenue = incremental_orders * second_order_aov
    gross_incremental_margin = (
        gross_incremental_revenue * row["contribution_margin_rate"]
    )
    cannibalised_margin = gross_incremental_margin * row["pull_forward_rate"]
    adjusted_incremental_margin = gross_incremental_margin - cannibalised_margin
    discount_cost = incremental_orders * row["discount_cost_per_order"]
    campaign_cost = eligible_customers * row["campaign_cost_per_customer"]
    net_incremental_value = adjusted_incremental_margin - discount_cost - campaign_cost
    revenue_efficiency = (
        gross_incremental_revenue / (discount_cost + campaign_cost)
        if (discount_cost + campaign_cost) > 0
        else np.nan
    )

    effective_margin_per_incremental_order = (
        second_order_aov
        * row["contribution_margin_rate"]
        * (1 - row["pull_forward_rate"])
        - row["discount_cost_per_order"]
    )
    break_even_incremental_conversion = (
        campaign_cost / (eligible_customers * effective_margin_per_incremental_order)
        if effective_margin_per_incremental_order > 0
        else np.nan
    )

    return pd.Series(
        {
            "eligible_customers": eligible_customers,
            "control_conversion": control_conversion,
            "treatment_conversion": row["treatment_conversion"],
            "incremental_conversion": incremental_conversion,
            "incremental_orders": incremental_orders,
            "gross_incremental_revenue": gross_incremental_revenue,
            "gross_incremental_margin": gross_incremental_margin,
            "cannibalised_margin": cannibalised_margin,
            "adjusted_incremental_margin": adjusted_incremental_margin,
            "discount_cost": discount_cost,
            "campaign_cost": campaign_cost,
            "net_incremental_value": net_incremental_value,
            "revenue_efficiency": revenue_efficiency,
            "break_even_incremental_conversion": break_even_incremental_conversion,
        }
    )

In [ ]:
scenario_results = pd.concat(
    [
        scenario_assumptions[["scenario"]],
        scenario_assumptions.apply(calculate_incrementality, axis=1),
        scenario_assumptions[
            [
                "contribution_margin_rate",
                "discount_cost_per_order",
                "campaign_cost_per_customer",
                "pull_forward_rate",
            ]
        ],
    ],
    axis=1,
)

scenario_results["decision"] = np.select(
    [
        scenario_results["net_incremental_value"] <= 0,
        scenario_results["incremental_conversion"]
        < scenario_results["break_even_incremental_conversion"],
        scenario_results["pull_forward_rate"] >= 0.35,
    ],
    [
        "Do not scale",
        "Below break-even",
        "Test cautiously",
    ],
    default="Scale candidate",
)

scenario_results.to_csv(
    PROCESSED_DIR / "incrementality_scenarios.csv",
    index=False,
)

scenario_results

## Sensitivity Analysis

The sensitivity table shows how net incremental value changes across different conversion uplift and pull-forward assumptions using the base scenario economics. This helps identify how much uplift is needed before the campaign should scale.

In [ ]:
base = scenario_assumptions[scenario_assumptions["scenario"] == "Base"].iloc[0]
uplift_points = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07]
pull_forward_rates = [0.10, 0.20, 0.30, 0.40, 0.50]

sensitivity_rows = []
for uplift in uplift_points:
    for pull_forward_rate in pull_forward_rates:
        row = base.copy()
        row["treatment_conversion"] = control_conversion + uplift
        row["pull_forward_rate"] = pull_forward_rate
        result = calculate_incrementality(row)
        sensitivity_rows.append(
            {
                "incremental_conversion": uplift,
                "pull_forward_rate": pull_forward_rate,
                "net_incremental_value": result["net_incremental_value"],
                "incremental_orders": result["incremental_orders"],
                "gross_incremental_revenue": result["gross_incremental_revenue"],
            }
        )

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(
    PROCESSED_DIR / "incrementality_sensitivity.csv",
    index=False,
)

sensitivity.pivot_table(
    index="incremental_conversion",
    columns="pull_forward_rate",
    values="net_incremental_value",
)

## Commercial Interpretation

The campaign should not be scaled because treatment conversion is higher in isolation. It should be scaled only when conversion uplift exceeds the break-even threshold and the remaining value is still positive after pull-forward, discounts, and campaign cost.

In [ ]:
summary_view = scenario_results[
    [
        "scenario",
        "incremental_conversion",
        "incremental_orders",
        "gross_incremental_revenue",
        "net_incremental_value",
        "break_even_incremental_conversion",
        "decision",
    ]
].copy()

summary_view["incremental_conversion"] = summary_view["incremental_conversion"].map(
    lambda value: f"{value:.1%}"
)
summary_view["break_even_incremental_conversion"] = summary_view[
    "break_even_incremental_conversion"
].map(lambda value: f"{value:.1%}")
summary_view["incremental_orders"] = summary_view["incremental_orders"].round(1)
summary_view["gross_incremental_revenue"] = summary_view[
    "gross_incremental_revenue"
].round(2)
summary_view["net_incremental_value"] = summary_view[
    "net_incremental_value"
].round(2)

summary_view

## Output Validation

The final checks confirm that scenario and sensitivity outputs were created and contain the expected fields.

In [ ]:
expected_outputs = {
    "incrementality_scenarios.csv": {
        "scenario",
        "eligible_customers",
        "control_conversion",
        "treatment_conversion",
        "incremental_conversion",
        "net_incremental_value",
        "break_even_incremental_conversion",
        "decision",
    },
    "incrementality_sensitivity.csv": {
        "incremental_conversion",
        "pull_forward_rate",
        "net_incremental_value",
    },
}

for file_name, expected_columns in expected_outputs.items():
    path = PROCESSED_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")

    output = pd.read_csv(path)
    missing_columns = sorted(expected_columns - set(output.columns))
    if missing_columns:
        raise ValueError(f"{file_name} is missing columns: {missing_columns}")

    print(f"Validated {file_name}: {output.shape}")